# Партиционирование и хранение — 30 заданий

Практика на eBay; решений нет.

## Результаты обучения

После **Хранение Spark** вы должны объяснить transformation как plan, предсказать action/jobs/shuffle, связать schema/grain с результатом и доказать физическую эффективность через explain/UI/metrics.

## Ментальная модель исполнения

Spark partition — единица task, Hive partition — каталог. Parquet row groups/statistics обеспечивают pruning/pushdown, а число выходных файлов управляется распределением перед write.

```text
transformations → unresolved logical plan
       ↓ analysis (catalog/types)
   optimized logical plan (Catalyst)
       ↓ physical planning / AQE
job → stage → shuffle → stage
       tasks             tasks
       └──── executors ─────┘
```
Action создаёт job. Один notebook/application может породить много jobs, а один job —
несколько stages. `repartition`, join и groupBy часто добавляют Exchange.

## Данные eBay

Grain eBay — `itemid` в `snapshot_dt`; 2 501 511 строк, 24 колонки, Parquet/Snappy.
Partition column — дата снимка. Цена, продавец, категории и доставка денормализованы.
Перед `latest item` или dedup проверяйте уникальность пары и задавайте tie-breaker.

Полная схема и проверки качества находятся в `data-catalog`. Raw read-only, результаты — в личном `spark_training`.

## Алгоритм решения

1. Зафиксируйте входной и целевой grain. 2. Выберите только нужные columns/rows. 3. Соберите transformation без action. 4. Проверьте schema и explain. 5. Предскажите partitions/shuffle. 6. Выполните минимальный action/write. 7. Повторно прочитайте и сверяйте keys/metrics. 8. Сохраните evidence.

Измеряйте input/output files, partitions, bytes, codec, schema и scan plan; не судите по имени метода.

## Типичные ошибки

- Вызывать count/show после каждого шага и создавать лишние jobs.
- Использовать Python UDF при наличии встроенной функции.
- Делать repartition без понимания Exchange и целевого файла.
- Broadcast большой стороны или collect на driver.
- Кэшировать одноразовый DataFrame без materialization/unpersist.
- Измерять скорость при разных результатах или непрогретом JVM.

## Самопроверка

1. Какой action создаёт job? 2. Где появится shuffle? 3. Сколько input/output partitions? 4. Видит ли Catalyst выражение? 5. Каков grain после JOIN/window? 6. Как проверить idempotent rerun?

## Подробная теория

### 1. Два смысла partition

Spark partition — task input, Hive partition — каталог по значению ключа.

### 2. Файлы

Мелкие файлы нагружают metadata/planning, слишком крупные ограничивают parallelism.

### 3. Parquet

Column pruning и row-group statistics уменьшают I/O.

### 4. Pruning

Явный фильтр partition column исключает каталоги до scan.

### 5. Запись

Append должен быть защищён от повтора; overwrite ограничивают нужной partition.

## Сдача

Каждое задание записывает непустой Parquet в личный HDFS и evidence с transformation, observation и explanation. Checker использует активную SparkSession.

In [ ]:
import os,sys
sys.path.insert(0,'/opt/lab/spark-training')
from check_task import check_task,save_evidence
from pyspark.sql import SparkSession,functions as F,types as T,Window
spark=SparkSession.builder.appName('spark-training').enableHiveSupport().getOrCreate()
USER=os.environ.get('HDFS_USER',os.environ.get('HADOOP_USER_NAME','student'))
ROOT=f'hdfs://namenode:8020/user/{USER}/spark_training'
SOURCE='hdfs://namenode:8020/data/raw/ebay'
ebay=spark.read.parquet(SOURCE)
print('Spark',spark.version,'rows',ebay.count(),'columns',len(ebay.columns))

### Задание 1. logical partitions

Создайте результат по теме **logical partitions** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_01")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',1)

### Задание 2. repartition N

Создайте результат по теме **repartition N** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_02")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',2)

### Задание 3. repartition key

Создайте результат по теме **repartition key** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_03")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',3)

### Задание 4. coalesce

Создайте результат по теме **coalesce** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_04")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',4)

### Задание 5. partitionBy write

Создайте результат по теме **partitionBy write** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_05")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',5)

### Задание 6. partition discovery

Создайте результат по теме **partition discovery** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_06")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',6)

### Задание 7. partition pruning

Создайте результат по теме **partition pruning** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_07")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',7)

### Задание 8. input files

Создайте результат по теме **input files** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_08")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',8)

### Задание 9. file count

Создайте результат по теме **file count** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_09")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',9)

### Задание 10. small files

Создайте результат по теме **small files** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_10")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',10)

### Задание 11. target size

Создайте результат по теме **target size** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_11")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',11)

### Задание 12. maxRecordsPerFile

Создайте результат по теме **maxRecordsPerFile** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_12")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',12)

### Задание 13. Snappy

Создайте результат по теме **Snappy** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_13")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',13)

### Задание 14. Gzip

Создайте результат по теме **Gzip** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_14")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',14)

### Задание 15. Parquet schema

Создайте результат по теме **Parquet schema** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_15")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',15)

### Задание 16. mergeSchema

Создайте результат по теме **mergeSchema** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_16")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',16)

### Задание 17. column pruning

Создайте результат по теме **column pruning** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_17")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',17)

### Задание 18. predicate pushdown

Создайте результат по теме **predicate pushdown** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_18")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',18)

### Задание 19. row group stats

Создайте результат по теме **row group stats** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_19")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',19)

### Задание 20. overwrite mode

Создайте результат по теме **overwrite mode** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_20")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',20)

### Задание 21. append mode

Создайте результат по теме **append mode** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_21")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',21)

### Задание 22. dynamic overwrite

Создайте результат по теме **dynamic overwrite** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_22")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',22)

### Задание 23. saveAsTable

Создайте результат по теме **saveAsTable** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_23")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',23)

### Задание 24. insertInto

Создайте результат по теме **insertInto** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_24")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',24)

### Задание 25. bucketBy

Создайте результат по теме **bucketBy** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_25")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',25)

### Задание 26. sortBy

Создайте результат по теме **sortBy** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_26")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',26)

### Задание 27. compaction

Создайте результат по теме **compaction** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_27")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',27)

### Задание 28. schema evolution

Создайте результат по теме **schema evolution** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_28")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',28)

### Задание 29. storage metrics

Создайте результат по теме **storage metrics** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_29")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',29)

### Задание 30. optimized dataset

Создайте результат по теме **optimized dataset** и запишите `mode("overwrite").parquet(f"{ROOT}/storage/task_30")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'storage',30)